# Table S4

Calculates drought-deficit offsets at a 20% pumping-reduction budget.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = next(p.resolve() for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'reconstruct.py').exists())
INPUT = ROOT / 'outputs' / 'MANAGEMENT_2012_2013' / 'equal_volume' / 'drought_strategy_seed_metrics.csv'
OUTPUT = ROOT / 'outputs' / 'tables' / 'TableS4.csv'
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

BUDGET = 20
PI75_FACTOR = 1.150349
DROUGHT_DEFICIT_M3 = 14849654274.015465
STRATEGIES = ['Uniform', 'High pumping', 'Leverage guided']

data = pd.read_csv(INPUT)
data = data.loc[data['budget_nominal'].eq(BUDGET) & data['strategy'].isin(STRATEGIES)].copy()

def mean_interval(values, scale=1.0, digits=2):
    values = np.asarray(values, dtype=float) / scale
    mean = float(values.mean())
    spread = PI75_FACTOR * float(values.std(ddof=0))
    return f'{mean:.{digits}f} [{mean - spread:.{digits}f}, {mean + spread:.{digits}f}]'

uniform = data.loc[data['strategy'].eq('Uniform'), ['seed', 'benefit_201212_m3']].rename(columns={'benefit_201212_m3': 'uniform_benefit'})
rows = []
for strategy in STRATEGIES:
    subset = data.loc[data['strategy'].eq(strategy)].copy()
    label = 'Leverage-guided' if strategy == 'Leverage guided' else strategy
    paired = subset.merge(uniform, on='seed', how='left', validate='one_to_one')
    additional_offset = 100 * (paired['benefit_201212_m3'] - paired['uniform_benefit']) / DROUGHT_DEFICIT_M3
    rows.append({
        'Strategy': label,
        'Actual pumping reduction (10^9 m^3)': f'{subset.actual_dV_m3.mean() / 1e9:.2f}',
        'Benefit, Oct 2012 (10^9 m^3; mean [75% interval])': mean_interval(subset['benefit_201210_m3'], scale=1e9),
        'Benefit, Dec 2012 (10^9 m^3; mean [75% interval])': mean_interval(subset['benefit_201212_m3'], scale=1e9),
        'Deficit offset, Dec 2012 (%; mean [75% interval])': mean_interval(100 * subset['benefit_201212_m3'] / DROUGHT_DEFICIT_M3),
        'Benefit, Apr 2013 (10^9 m^3; mean [75% interval])': mean_interval(subset['benefit_201304_m3'], scale=1e9),
        'Additional deficit offset vs Uniform (percentage points; mean [75% interval])': '—' if strategy == 'Uniform' else mean_interval(additional_offset),
    })

table = pd.DataFrame(rows)
table.to_csv(OUTPUT, index=False)
table
